Amna Rafique

## Final Assignment 

 For the obligatory assignment, you will use what you learned so far. The task is to code the solutions for the exercises, as well as edit the markdown cells underneath them, reporting  your choices, findings, and interpretations.

## A random protein dataset
You will work with a new randomly generated dataset called `mice_dataset.csv`. There is a cell with the code to set up the dataset, but be careful to only run that code once, otherwise you will overwrite your previous dataset, and your conclusions and results might not be reliable anymore.

## The goal

The evaluation __is not__ related to the results of your analysis. We are not looking for p-values or coefficients.
We will evaluate if: 
1. you can deliver a notebook that can run without `error`. 
2. you are applying the right tools. 
3. you can report your reasoning clearly.

## How to deliver
On GitHub, create a repository and invite us (HenriChap, JM-Godoy, hbarsnes) to contribute, so that we can evaluate.
Remember to set the visibility of your repo to `private` and write a `README.md` file.

---
### Setting up the exercise (generating random data)

Run the code below __only once__ to start your assignment.

Suggestion: comment out the code using `#` to make sure it won't run again.


In [22]:
#from random_dataset_generator import save_random_mice_dataset

#df = save_random_mice_dataset()

### Loading resources needed
Load all packages and modules needed. A nice way to keep track of the packages is adding them app along you are using them.

In [6]:
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy import stats
import statsmodels.formula.api as smf

if not pathlib.Path("figures").exists():
    pathlib.Path("figures").mkdir()

sns.set_theme(style="whitegrid", context="notebook")

### Reading the file

Load the `mice_dataset.csv`.

In [7]:
data_folder_path = pathlib.Path ("..", "Final_Assignment","mice_dataset.csv") 
df = pd.read_csv(data_folder_path)
df.head()

,ID,Sex,Group,Weight(g),Age(weeks),Protein1,Protein2,Protein3,Protein4,Protein5,Protein6
0,Mice001,F,Control,24.991485,28.0,87.470986,145.572388,19.542352,977.994262,184.939206,223.871958
1,Mice002,M,Treatment,29.598742,34.0,103.181890,233.686016,NaN,334.015484,247.919290,405.974712
2,Mice003,F,Treatment,20.745661,30.0,82.089572,147.438270,NaN,797.238531,177.627953,326.103146
3,Mice004,M,Control,27.602722,27.0,105.358525,142.436325,NaN,820.147673,222.082639,235.719952
4,Mice005,F,Treatment,23.012260,31.0,88.398401,196.417504,NaN,45.059788,209.557178,318.914550


Pathlib is a module that is used when working with folder and reading files. It helps ensure that the file paths are formatted correctly and checks if the path exist or not, instead of writing file address as a plain text string in my coding. in line 1, i wrote ("..", "Final_Assignment","mice_dataset.csv"), even tho the file of interest is the mice dataset, but writing the folder too helps to locate it eaiser than writing ony the dataset. I encountered an error where the file was not found , that`s why i choose to input it like so. "df.head ()" is a function that shows the forst 5 rows of the dataframe. This is useful to get a quick overview of the data and see if its loaded correctly. 

### Describe the data 

In [25]:
df.describe()

,Weight(g),Age(weeks),Protein1,Protein2,Protein3,Protein4,Protein5,Protein6
count,138.000000,137.00000,138.000000,137.000000,27.000000,137.000000,138.000000,138.000000
mean,24.784582,30.10219,108.563398,159.335186,28.248701,492.518990,225.719706,311.120211
std,3.830300,2.49495,14.928933,27.506250,5.474753,283.020030,24.786657,51.246307
min,17.154252,26.00000,76.663634,94.432761,17.983362,18.557439,164.351776,190.409751
25%,21.471427,28.00000,96.412762,142.158117,25.120396,258.174400,209.834959,274.775324
50%,24.724618,30.00000,110.001274,157.064400,27.918421,513.281828,223.516257,314.082957
75%,28.506990,32.00000,120.250872,179.119575,32.471382,704.388769,243.549884,350.780864
max,32.001276,34.00000,139.926816,233.686016,36.552499,995.883561,284.571094,408.751484


The code i chose to run provides a statistical summary of the numbered variables in the datasaet rather than using df.info (which gives information about the structure of the dataset). So now i can understand the distribution and range of the data.
Count: measures the sample size. The total number of valid observations and since some columns have lower count than others, that means some values are missing. This is very visible for protein3 where count is 27 (i`ll inspect this later). 
Mean: the average value. The number gives an overall general value of each category (weight, age etc.).
Std (Standard deviation): Tells me how far the value strays from eachother. so a low std means that the data are clustered to an average point, but a high std means the values are widely spread out (spread/dispersion).
Minimum: tells me the lowest data value recorded in each column.
25% (1st Quartile): the numerical value here shows that 25% of the observations fall below it (the value i mean) example, 25% of the weight recorded falls below 21.47 grams. 
50% (2nd Quartile): same concept, but tells the exact middle point the data when sorted from lowest to highest, so half are below this numerical value, while the other half is above. 
75% (3rd Quartile): Same gist, 75% of all observations fall below this numerical value.
Maximum: tells me the highest data value recorded in each column.


### Inspect and clean your data 

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ID          140 non-null    str    
 1   Sex         137 non-null    str    
 2   Group       139 non-null    str    
 3   Weight(g)   138 non-null    float64
 4   Age(weeks)  137 non-null    float64
 5   Protein1    138 non-null    float64
 6   Protein2    137 non-null    float64
 7   Protein3    27 non-null     float64
 8   Protein4    137 non-null    float64
 9   Protein5    138 non-null    float64
 10  Protein6    138 non-null    float64
dtypes: float64(8), str(3)
memory usage: 14.4 KB


In [9]:
df.isnull().sum()

ID              0
Sex             3
Group           1
Weight(g)       2
Age(weeks)      3
Protein1        2
Protein2        3
Protein3      113
Protein4        3
Protein5        2
Protein6        2
dtype: int64

In [10]:
df.duplicated().sum()

np.int64(0)

In [24]:
df.dropna()

,ID,Sex,Group,Weight(g),Age(weeks),Protein1,Protein2,Protein4,Protein5,Protein6
0,Mice001,F,Control,24.991485,28.0,87.470986,145.572388,977.994262,184.939206,223.871958
1,Mice002,M,Treatment,29.598742,34.0,103.181890,233.686016,334.015484,247.919290,405.974712
2,Mice003,F,Treatment,20.745661,30.0,82.089572,147.438270,797.238531,177.627953,326.103146
3,Mice004,M,Control,27.602722,27.0,105.358525,142.436325,820.147673,222.082639,235.719952
4,Mice005,F,Treatment,23.012260,31.0,88.398401,196.417504,45.059788,209.557178,318.914550
...,...,...,...,...,...,...,...,...,...,...
135,Mice136,F,Control,20.579871,31.0,84.029261,153.889688,596.317074,188.072118,307.990454
136,Mice137,M,Treatment,26.392027,28.0,105.396252,139.984073,71.344992,217.523685,375.985551
137,Mice138,M,Treatment,25.346787,32.0,118.901958,198.548741,653.903912,256.077616,307.333298
138,Mice139,F,Control,18.620937,28.0,108.951376,144.467482,344.752841,224.343790,272.399782


In [25]:
mice_clean = df.dropna()
print(mice_clean)

          ID Sex      Group  Weight(g)  Age(weeks)    Protein1    Protein2  \
0    Mice001   F    Control  24.991485        28.0   87.470986  145.572388   
1    Mice002   M  Treatment  29.598742        34.0  103.181890  233.686016   
2    Mice003   F  Treatment  20.745661        30.0   82.089572  147.438270   
3    Mice004   M    Control  27.602722        27.0  105.358525  142.436325   
4    Mice005   F  Treatment  23.012260        31.0   88.398401  196.417504   
..       ...  ..        ...        ...         ...         ...         ...   
135  Mice136   F    Control  20.579871        31.0   84.029261  153.889688   
136  Mice137   M  Treatment  26.392027        28.0  105.396252  139.984073   
137  Mice138   M  Treatment  25.346787        32.0  118.901958  198.548741   
138  Mice139   F    Control  18.620937        28.0  108.951376  144.467482   
139  Mice140   M    Control  29.222933        32.0  125.098996  165.413577   

       Protein4    Protein5    Protein6  
0    977.994262  184.

In [26]:
mice_clean.to_csv("mice_clean.csv", index = False)

Data inspection and cleaning i where i can check for any missing values (NaNs), incorrect data types or duplicate entries. I typed in function df.info to get a quick summary of row counts, column names, data types and the non-null counts. As visible, the non-null count for column: protein 3, is way lower than the total number of rows (entries). This informs me of all the NaN here. There are more missing data too in the other columns, so to get en exact count i used isnull.sum, and counts the exact missing values in each column. Since there is quite a lot of numerical data, i wanted to check if there weren`t any duplicated data and the results shows that there is none duplication in my dataframe. After inspection, i chose to clean the dataset by removing the rows/columns with missing value. Since i saw that one column had 113 NaN, it is better to drop it from the dataframe than trying to impute all that missing data. removing them will also preserve the integrity of the dataset. Since only three rows had NaN, Removing them leaves me with 137 entries with complete observations without reducing the data considerably. All the "cleaning" part is done by using the method df.dropna. Since i found the column names quite clarifying, i dont see the necessity to be changing them. I thought of only changing the protein names cause protein3 column is now removed and there are only 5 protein columns in the new dataframe. i refrained from doing this in case the numbers are significant. 

### Understand the mice population.
Use table and plots to understand how is your data distributed.

In [30]:
print(mice_clean["Group"].value_counts())

Group
Treatment    69
Control      68
Name: count, dtype: int64


In [31]:
print(mice_clean["Sex"].value_counts())

Sex
F    69
M    68
Name: count, dtype: int64


Your notes: Frequency count for a categorical column, here the group. Tells me how many mices are in control and treatment group each. 

### Explore the relationship between the proteins

Plot data to give you information about possible relationships between proteins.

Your notes:

### Choose two proteins and explore the relationship between them.

After plotting the data, use linear regression to understand the relationship of those two proteins.

Your notes:

### Choose a protein and test if there is a difference between any non-protein variable

Your notes: